# Lineups Monte Carlo Validation

This notebook validates the game engine separately from the gameplay notebook. It runs randomized simulations, reports completion behavior, and includes targeted rule checks.

In [9]:
import copy
import json
import random
import statistics
from collections import Counter, deque

import pandas as pd
from IPython.display import display


In [10]:
with open('eligible_players.json', 'r') as f:
    eligible_players = json.load(f)['players']

with open('allPlayByPlay_data.json', 'r') as f:
    all_play_by_play = json.load(f)

print(f'Eligible players loaded: {len(eligible_players)}')
print(f'Games loaded with play-by-play: {len(all_play_by_play)}')


Eligible players loaded: 499
Games loaded with play-by-play: 13


In [11]:
def map_play_to_event(result_raw, description):
    r = (result_raw or '').strip().lower()
    d = (description or '').strip().lower()

    if 'pitching change' in d:
        return None, None
    if 'wild pitch' in d:
        return None, None
    if 'steals' in d or 'stolen base' in d:
        return None, None
    if r in {'caught stealing 2b', 'caught stealing home'}:
        return None, None

    if 'home run' in r:
        return 'home_run', None
    if r == 'triple':
        return 'triple', None
    if r == 'double':
        return 'double', None
    if r == 'single':
        return 'single', None
    if r in {'walk', 'intent walk', 'hit by pitch'}:
        return 'walk', None
    if r == 'field error':
        return 'reach_on_error', None

    fly_out_results = {'flyout', 'lineout', 'pop out', 'bunt pop out', 'sac fly'}
    ground_out_results = {
        'groundout', 'bunt groundout', 'forceout', 'grounded into dp',
        'double play', 'strikeout double play', 'fielders choice',
        'fielders choice out', 'sac bunt'
    }

    if r in fly_out_results:
        return 'out', 'fly'
    if r in ground_out_results:
        return 'out', 'ground'
    if r == 'strikeout':
        return 'out', 'other'

    if 'grounds out' in d or "fielder's choice" in d:
        return 'out', 'ground'
    if 'flies out' in d or 'lines out' in d or 'pops out' in d:
        return 'out', 'fly'
    if 'strikes out' in d:
        return 'out', 'other'
    if 'walks' in d or 'intentionally walks' in d or 'hit by pitch' in d:
        return 'walk', None
    if 'homers' in d:
        return 'home_run', None
    if 'triples' in d:
        return 'triple', None
    if 'doubles' in d:
        return 'double', None
    if 'singles' in d:
        return 'single', None

    return 'unknown', 'other'


def normalize_play_by_play(raw_games):
    rows = []
    filtered = 0

    for game in raw_games:
        game_id = game.get('gameID')
        plays = game.get('allPlayByPlay') or []
        for idx, play in enumerate(plays, start=1):
            player_id = play.get('playerID')
            if not player_id:
                filtered += 1
                continue

            event_type, out_kind = map_play_to_event(play.get('result'), play.get('description'))
            if event_type is None:
                filtered += 1
                continue

            rows.append({
                'game_id': game_id,
                'sequence_number': idx,
                'player_id': str(player_id),
                'result_raw': play.get('result'),
                'description': play.get('description'),
                'event_type': event_type,
                'out_kind': out_kind,
                'inning': play.get('inning')
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['game_id', 'sequence_number']).reset_index(drop=True)

    return df, filtered


plays_df, filtered_count = normalize_play_by_play(all_play_by_play)
playable_player_ids = set(plays_df['player_id'].astype(str).unique())
monte_carlo_eligible_players = [
    player for player in eligible_players
    if str(player['playerID']) in playable_player_ids
]

print(f'Normalized plays: {len(plays_df)}')
print(f'Filtered plays: {filtered_count}')
print(f'Monte Carlo eligible players with at least one play: {len(monte_carlo_eligible_players)}')
display(plays_df['event_type'].value_counts(dropna=False).rename_axis('event_type').reset_index(name='count'))


Normalized plays: 832
Filtered plays: 101
Monte Carlo eligible players with at least one play: 253


,event_type,count
0,out,597
1,single,111
2,walk,71
3,double,24
4,home_run,21
5,reach_on_error,7
6,triple,1


In [12]:
class FantasyGame:
    def __init__(self, verbose=False):
        self.inning = 1
        self.outs = 0
        self.bases = {'1B': None, '2B': None, '3B': None}
        self.score = 0
        self.current_batter = None
        self.game_over = False
        self.log = []
        self.verbose = verbose
        self.sac_fly_count = 0
        self.double_play_count = 0
        self.termination_reason = None

    def record(self, message):
        if self.verbose:
            print(message)
        self.log.append(message)

    def print_state(self):
        self.record(
            f"Inning: {self.inning} | Outs: {self.outs} | Bases: {self.bases} | Score: {self.score}"
        )

    def hit(self, bases_advanced, label='hits'):
        self.record(f"{self.current_batter['name']} {label}.")

        if bases_advanced == 1 and self.bases['2B']:
            runner = self.bases['2B']
            self.record(f"{runner['name']} scores from second on the single.")
            self.bases['2B'] = None
            self.score += 1

        runs_scored = []
        for base in reversed(range(1, 4)):
            runner = self.bases[f'{base}B']
            if runner is not None:
                new_base = base + bases_advanced
                if new_base > 3:
                    self.score += 1
                    runs_scored.append(runner['name'])
                else:
                    self.bases[f'{new_base}B'] = runner
                self.bases[f'{base}B'] = None

        if bases_advanced > 3:
            self.score += 1
            runs_scored.append(self.current_batter['name'])
        else:
            self.bases[f'{bases_advanced}B'] = self.current_batter

        if runs_scored:
            self.record(', '.join(runs_scored) + ' score.')

    def walk(self):
        self.record(f"{self.current_batter['name']} walks.")
        if self.bases['1B'] and self.bases['2B'] and self.bases['3B']:
            self.score += 1
            self.record(f"{self.bases['3B']['name']} scores.")
            self.bases['3B'] = self.bases['2B']
            self.bases['2B'] = self.bases['1B']
        elif self.bases['1B'] and self.bases['2B']:
            self.bases['3B'] = self.bases['2B']
            self.bases['2B'] = self.bases['1B']
        elif self.bases['1B']:
            self.bases['2B'] = self.bases['1B']
        self.bases['1B'] = self.current_batter

    def out(self):
        self.outs += 1
        if self.outs >= 3:
            self.print_state()
            self.outs = 0
            self.inning += 1
            self.bases = {'1B': None, '2B': None, '3B': None}
            if self.inning > 9:
                self.game_over = True
                self.termination_reason = 'completed_9'

    def double_play(self):
        self.double_play_count += 1
        if self.bases['1B'] and self.bases['2B'] and self.bases['3B']:
            self.record(
                f"{self.current_batter['name']} grounds into double play. {self.bases['3B']['name']} out at home."
            )
            self.bases['3B'] = self.bases['2B']
            self.bases['2B'] = self.bases['1B']
            self.bases['1B'] = None
        else:
            if self.bases['1B']:
                self.record(
                    f"{self.current_batter['name']} grounds into double play. {self.bases['1B']['name']} out at 2B."
                )
                self.bases['1B'] = None
            else:
                self.record(f"{self.current_batter['name']} grounds into double play.")
            if self.bases['2B']:
                self.bases['3B'] = self.bases['2B']
                self.bases['2B'] = None
        self.out()
        self.out()

    def sac_fly(self):
        self.sac_fly_count += 1
        if self.bases['3B'] and self.outs < 2:
            self.record(
                f"{self.current_batter['name']} flies out, {self.bases['3B']['name']} tags and scores."
            )
            self.score += 1
            self.bases['3B'] = None
        else:
            self.record(f"{self.current_batter['name']} flies out.")
        self.out()

    def process_event(self, event_type, result_raw='', out_kind='other'):
        self.record(f"{self.current_batter['name']} at bat: {result_raw} -> {event_type}")

        if event_type == 'single':
            self.hit(1, label='singles')
        elif event_type == 'double':
            self.hit(2, label='doubles')
        elif event_type == 'triple':
            self.hit(3, label='triples')
        elif event_type == 'home_run':
            self.hit(4, label='homers')
        elif event_type == 'walk':
            self.walk()
        elif event_type == 'reach_on_error':
            self.hit(1, label='reaches')
        elif event_type == 'out':
            if self.outs < 2 and self.bases['1B'] and out_kind == 'ground':
                self.double_play()
            elif self.outs < 2 and self.bases['3B'] and out_kind == 'fly':
                self.sac_fly()
            else:
                self.record(f"{self.current_batter['name']} is out.")
                self.out()
        else:
            self.record(f"{self.current_batter['name']} is out.")
            self.out()

        self.print_state()


In [13]:
def build_player_play_map(df):
    player_play_map = {}
    for player_id, group in df.groupby('player_id', sort=False):
        first_game = group['game_id'].iloc[0]
        player_rows = group[group['game_id'] == first_game].sort_values('sequence_number')
        player_play_map[player_id] = player_rows.to_dict(orient='records')
    return player_play_map


def validate_setup(selected_players):
    starter_slots = []
    names = []

    for pos in ['C', '1B', '2B', '3B', 'SS', 'DH']:
        player = selected_players[pos]
        if not player.get('id') or not player.get('lineup_slot'):
            return False, f'Missing starter selection or lineup slot at {pos}.'
        starter_slots.append(player['lineup_slot'])
        names.append(player['name'])

    for idx, player in enumerate(selected_players['OF'], start=1):
        if not player.get('id') or not player.get('lineup_slot'):
            return False, f'Missing OF{idx} selection or lineup slot.'
        starter_slots.append(player['lineup_slot'])
        names.append(player['name'])

    if sorted(starter_slots) != list(range(1, 10)):
        return False, 'Lineup slots must be exactly 1-9 with no duplicates.'

    ph_names = [player['name'] for player in selected_players['PH'] if player.get('id')]
    names.extend(ph_names)
    if len(names) != len(set(names)):
        return False, 'Duplicate player found between starters and bench.'

    return True, 'OK'


def build_lineup_and_bench(selected_players):
    starters = []
    for pos in ['C', '1B', '2B', '3B', 'SS', 'DH']:
        player = selected_players[pos]
        if player.get('lineup_slot'):
            starters.append((pos, copy.deepcopy(player)))
    for player in selected_players['OF']:
        if player.get('lineup_slot'):
            starters.append(('OF', copy.deepcopy(player)))

    starters = deque(sorted(starters, key=lambda item: item[1]['lineup_slot']))
    ph = deque(sorted(
        [copy.deepcopy(player) for player in selected_players['PH'] if player.get('id')],
        key=lambda item: item['bench_order']
    ))
    return starters, ph


def summarize_selected_players(selected_players):
    starters = []
    for pos in ['C', '1B', '2B', '3B', 'SS', 'DH']:
        starters.append(selected_players[pos])
    starters.extend(selected_players['OF'])
    starters = sorted(starters, key=lambda player: player['lineup_slot'])
    starter_names = ', '.join(player['name'] for player in starters)
    bench_names = ', '.join(
        player['name'] for player in sorted(selected_players['PH'], key=lambda player: player.get('bench_order', 999))
        if player.get('id')
    )
    return starter_names, bench_names


def run_single_simulation(selected_players, plays_df, verbose=False):
    selected_players = copy.deepcopy(selected_players)
    valid, message = validate_setup(selected_players)
    if not valid:
        return {
            'completed_nine': False,
            'termination_reason': 'invalid_setup',
            'final_inning': None,
            'final_outs': None,
            'score': None,
            'sac_flies': 0,
            'double_plays': 0,
            'log_length': 0,
            'remaining_lineup_spots': 0,
            'remaining_ph': 0,
            'selected_players': selected_players,
            'game': None,
            'validation_message': message
        }

    play_map = build_player_play_map(plays_df)
    lineup_queue, ph_queue = build_lineup_and_bench(selected_players)
    game = FantasyGame(verbose=verbose)
    play_index = {}

    for _, player in lineup_queue:
        play_index.setdefault(str(player['id']), 0)
    for player in ph_queue:
        play_index.setdefault(str(player['id']), 0)

    try:
        while not game.game_over:
            if not lineup_queue:
                game.termination_reason = 'no_players_left'
                break

            processed_at_bat = 0
            turns = len(lineup_queue)

            for _ in range(turns):
                if game.game_over:
                    break

                pos, player_info = lineup_queue.popleft()
                pid = str(player_info['id'])
                batter_plays = play_map.get(pid, [])
                idx = play_index.get(pid, 0)

                if idx >= len(batter_plays):
                    if ph_queue:
                        sub = ph_queue.popleft()
                        game.record(f"Subbing in {sub['name']} for {player_info['name']} at {pos}.")
                        player_info = sub
                        pid = str(player_info['id'])
                        batter_plays = play_map.get(pid, [])
                        idx = play_index.get(pid, 0)
                    else:
                        game.record(f"{player_info['name']} has no remaining at-bats and no PH available.")
                        continue

                if idx < len(batter_plays):
                    play = batter_plays[idx]
                    game.current_batter = player_info
                    game.process_event(
                        play['event_type'],
                        play.get('result_raw', ''),
                        play.get('out_kind', 'other')
                    )
                    play_index[pid] = idx + 1
                    lineup_queue.append((pos, player_info))
                    processed_at_bat += 1
                else:
                    game.record(f"{player_info['name']} has no usable plays. Skipping.")

            if not game.game_over and processed_at_bat == 0:
                game.termination_reason = 'no_usable_batters'
                break

        if game.game_over and game.termination_reason is None:
            game.termination_reason = 'completed_9'

    except Exception as exc:
        game.termination_reason = 'error'
        game.record(f'Runtime error: {exc}')

    completed_nine = bool(game.game_over and game.inning > 9 and game.termination_reason == 'completed_9')
    return {
        'completed_nine': completed_nine,
        'termination_reason': game.termination_reason,
        'final_inning': game.inning,
        'final_outs': game.outs,
        'score': game.score,
        'sac_flies': game.sac_fly_count,
        'double_plays': game.double_play_count,
        'log_length': len(game.log),
        'remaining_lineup_spots': len(lineup_queue),
        'remaining_ph': len(ph_queue),
        'selected_players': selected_players,
        'game': game,
        'validation_message': message
    }


In [14]:
def generate_random_selected_players(eligible_players, rng):
    players_by_pos = {}
    for pos in ['C', '1B', '2B', '3B', 'SS', 'OF', 'DH']:
        players_by_pos[pos] = [player for player in eligible_players if player['pos'] == pos]

    selected_ids = set()
    selected_players = {
        'C': {'id': '', 'name': '', 'lineup_slot': ''},
        '1B': {'id': '', 'name': '', 'lineup_slot': ''},
        '2B': {'id': '', 'name': '', 'lineup_slot': ''},
        '3B': {'id': '', 'name': '', 'lineup_slot': ''},
        'SS': {'id': '', 'name': '', 'lineup_slot': ''},
        'OF': [{'id': '', 'name': '', 'lineup_slot': ''} for _ in range(3)],
        'DH': {'id': '', 'name': '', 'lineup_slot': ''},
        'PH': []
    }

    for pos in ['C', '1B', '2B', '3B', 'SS', 'DH']:
        player = rng.choice(players_by_pos[pos])
        while player['playerID'] in selected_ids:
            player = rng.choice(players_by_pos[pos])
        selected_ids.add(player['playerID'])
        selected_players[pos] = {
            'id': str(player['playerID']),
            'name': player['longName'],
            'lineup_slot': ''
        }

    of_choices = rng.sample([player for player in players_by_pos['OF'] if player['playerID'] not in selected_ids], 3)
    for idx, player in enumerate(of_choices):
        selected_ids.add(player['playerID'])
        selected_players['OF'][idx] = {
            'id': str(player['playerID']),
            'name': player['longName'],
            'lineup_slot': ''
        }

    bench_pool = [player for player in eligible_players if player['pos'] != 'P' and player['playerID'] not in selected_ids]
    for idx, player in enumerate(rng.sample(bench_pool, 6), start=1):
        selected_players['PH'].append({
            'id': str(player['playerID']),
            'name': player['longName'],
            'lineup_slot': '',
            'bench_order': idx,
            'position': 'PH'
        })

    starters = []
    for pos in ['C', '1B', '2B', '3B', 'SS', 'DH']:
        starters.append(selected_players[pos])
    starters.extend(selected_players['OF'])
    rng.shuffle(starters)
    for idx, player in enumerate(starters, start=1):
        player['lineup_slot'] = idx

    return selected_players


def run_random_simulations(num_games=100, seed=42):
    rng = random.Random(seed)
    rows = []

    for simulation_id in range(1, num_games + 1):
        selected_players = generate_random_selected_players(monte_carlo_eligible_players, rng)
        result = run_single_simulation(selected_players, plays_df, verbose=False)
        starter_names, bench_names = summarize_selected_players(selected_players)
        rows.append({
            'simulation_id': simulation_id,
            'completed_nine': result['completed_nine'],
            'termination_reason': result['termination_reason'],
            'final_inning': result['final_inning'],
            'final_outs': result['final_outs'],
            'score': result['score'],
            'sac_flies': result['sac_flies'],
            'double_plays': result['double_plays'],
            'remaining_lineup_spots': result['remaining_lineup_spots'],
            'remaining_ph': result['remaining_ph'],
            'starter_names': starter_names,
            'bench_names': bench_names
        })

    return pd.DataFrame(rows)


In [17]:
results_df = run_random_simulations(num_games=1000, seed=42)

games_simulated = len(results_df)
completed_games = int(results_df['completed_nine'].sum())
incomplete_games = games_simulated - completed_games
completion_rate = completed_games / games_simulated if games_simulated else 0.0
average_score = statistics.mean(results_df['score']) if games_simulated else 0.0
median_score = statistics.median(results_df['score']) if games_simulated else 0.0
mode_scores = statistics.multimode(results_df['score']) if games_simulated else []
total_sac_flies = int(results_df['sac_flies'].sum())
average_sac_flies_per_game = total_sac_flies / games_simulated if games_simulated else 0.0
total_double_plays = int(results_df['double_plays'].sum())
average_double_plays_per_game = total_double_plays / games_simulated if games_simulated else 0.0

summary_df = pd.DataFrame([
    {
        'games_simulated': games_simulated,
        'completed_games': completed_games,
        'incomplete_games': incomplete_games,
        'completion_rate': round(completion_rate, 4),
        'average_score': round(average_score, 2),
        'median_score': median_score,
        'mode_scores': ', '.join(str(value) for value in mode_scores),
        'total_sac_flies': total_sac_flies,
        'average_sac_flies_per_game': round(average_sac_flies_per_game, 3),
        'total_double_plays': total_double_plays,
        'average_double_plays_per_game': round(average_double_plays_per_game, 3)
    }
])

termination_reason_df = (
    results_df['termination_reason']
    .value_counts(dropna=False)
    .rename_axis('termination_reason')
    .reset_index(name='count')
)

flagged_games_df = (
    results_df.loc[~results_df['completed_nine'], [
        'simulation_id', 'termination_reason', 'final_inning', 'final_outs', 'score',
        'sac_flies', 'double_plays', 'starter_names', 'bench_names'
    ]]
    .sort_values(['final_inning', 'termination_reason', 'simulation_id'], na_position='first')
    .reset_index(drop=True)
)

incomplete_by_inning_df = (
    flagged_games_df['final_inning']
    .value_counts(dropna=False)
    .rename_axis('final_inning')
    .reset_index(name='count')
    .sort_values('final_inning', na_position='first')
)

roster_exhaustion_pct = round(
    100.0 * (results_df['termination_reason'] == 'no_usable_batters').sum() / incomplete_games,
    2
) if incomplete_games else 0.0

print(f'Games simulated: {games_simulated}')
print(f'Completed 9 innings: {completed_games}')
print(f'Did not complete: {incomplete_games}')
print(f'Completion rate: {completion_rate:.2%}')
print(f'Average score: {average_score:.2f}')
print(f'Median score: {median_score}')
print(f"Mode score(s): {', '.join(str(value) for value in mode_scores)}")
print(f'Total sac flies: {total_sac_flies}')
print(f'Total double plays: {total_double_plays}')
print(f'Incomplete games due to roster exhaustion: {roster_exhaustion_pct}%')

display(summary_df)
display(termination_reason_df)
display(incomplete_by_inning_df)
display(flagged_games_df)


Games simulated: 1000
Completed 9 innings: 992
Did not complete: 8
Completion rate: 99.20%
Average score: 2.92
Median score: 2.0
Mode score(s): 2
Total sac flies: 125
Total double plays: 2065
Incomplete games due to roster exhaustion: 100.0%


,games_simulated,completed_games,incomplete_games,completion_rate,average_score,median_score,mode_scores,total_sac_flies,average_sac_flies_per_game,total_double_plays,average_double_plays_per_game
0,1000,992,8,0.992,2.92,2.0,2,125,0.125,2065,2.065


,termination_reason,count
0,completed_9,992
1,no_usable_batters,8


,final_inning,count
1,8,2
0,9,6


,simulation_id,termination_reason,final_inning,final_outs,score,sac_flies,double_plays,starter_names,bench_names
0,269,no_usable_batters,8,2,9,0,1,"Miguel Rojas, Kameron Misner, Cal Raleigh, Mar...","CJ Abrams, Gabriel Moreno, Harrison Bader, Set..."
1,461,no_usable_batters,8,2,4,1,1,"Austin Wells, Anthony Santander, Kyren Paris, ...","Michael Massey, Michael A. Taylor, Mike Trout,..."
2,77,no_usable_batters,9,2,12,1,2,"Andrew Vaughn, Junior Caminero, Tim Anderson, ...","Jhonkensy Noel, Michael Massey, Victor Caratin..."
3,100,no_usable_batters,9,2,7,0,0,"Enrique Hernández, Nolan Jones, Luis Arraez, T...","Graham Pauley, Willson Contreras, Spencer Tork..."
4,442,no_usable_batters,9,0,10,0,1,"Edouard Julien, Luken Baker, Cal Raleigh, Jord...","LaMonte Wade Jr., Seth Brown, Lenyn Sosa, Jona..."
5,789,no_usable_batters,9,2,6,0,0,"Casey Schmitt, Dane Myers, Marcell Ozuna, Harr...","Lawrence Butler, Wyatt Langford, Oswaldo Cabre..."
6,875,no_usable_batters,9,1,11,0,0,"Matt Chapman, Jackson Merrill, Tyler Soderstro...","Nico Hoerner, CJ Abrams, Michael A. Taylor, Br..."
7,966,no_usable_batters,9,2,7,0,1,"Alan Roden, Marcell Ozuna, Nolan Arenado, Vict...","Ben Rice, Rowdy Tellez, Brandon Nimmo, Cavan B..."


In [16]:
def empty_selected_players_template():
    return {
        'C': {'id': '', 'name': '', 'lineup_slot': ''},
        '1B': {'id': '', 'name': '', 'lineup_slot': ''},
        '2B': {'id': '', 'name': '', 'lineup_slot': ''},
        '3B': {'id': '', 'name': '', 'lineup_slot': ''},
        'SS': {'id': '', 'name': '', 'lineup_slot': ''},
        'OF': [{'id': '', 'name': '', 'lineup_slot': ''} for _ in range(3)],
        'DH': {'id': '', 'name': '', 'lineup_slot': ''},
        'PH': []
    }


def build_fixture_selected_players(include_ph=True):
    selected_players = empty_selected_players_template()
    starters = [
        ('C', {'id': 's1', 'name': 'Starter 1', 'lineup_slot': 1}),
        ('1B', {'id': 's2', 'name': 'Starter 2', 'lineup_slot': 2}),
        ('2B', {'id': 's3', 'name': 'Starter 3', 'lineup_slot': 3}),
        ('3B', {'id': 's4', 'name': 'Starter 4', 'lineup_slot': 4}),
        ('SS', {'id': 's5', 'name': 'Starter 5', 'lineup_slot': 5}),
        ('DH', {'id': 's6', 'name': 'Starter 6', 'lineup_slot': 6})
    ]
    for pos, player in starters:
        selected_players[pos] = player

    selected_players['OF'] = [
        {'id': 's7', 'name': 'Starter 7', 'lineup_slot': 7},
        {'id': 's8', 'name': 'Starter 8', 'lineup_slot': 8},
        {'id': 's9', 'name': 'Starter 9', 'lineup_slot': 9}
    ]

    if include_ph:
        selected_players['PH'] = [
            {'id': 'ph1', 'name': 'Bench 1', 'lineup_slot': '', 'bench_order': 1, 'position': 'PH'}
        ]
    return selected_players


def run_targeted_engine_checks():
    results = []

    game = FantasyGame(verbose=False)
    game.current_batter = {'name': 'Batter'}
    game.bases['3B'] = {'name': 'Runner 3B'}
    game.process_event('out', 'Flyout', 'fly')
    results.append({
        'test_name': 'sac fly with runner on 3B and 0 outs',
        'passed': game.sac_fly_count == 1 and game.score == 1 and game.outs == 1,
        'expected': 'sac_fly_count=1, score=1, outs=1',
        'actual': f'sac_fly_count={game.sac_fly_count}, score={game.score}, outs={game.outs}'
    })

    game = FantasyGame(verbose=False)
    game.current_batter = {'name': 'Batter'}
    game.bases['1B'] = {'name': 'Runner 1B'}
    game.process_event('out', 'Groundout', 'ground')
    results.append({
        'test_name': 'double play with runner on 1B and 0 outs',
        'passed': game.double_play_count == 1 and game.outs == 2,
        'expected': 'double_play_count=1, outs=2',
        'actual': f'double_play_count={game.double_play_count}, outs={game.outs}'
    })

    game = FantasyGame(verbose=False)
    game.current_batter = {'name': 'Batter'}
    game.bases['1B'] = {'name': 'Runner 1B'}
    game.outs = 2
    game.process_event('out', 'Groundout', 'ground')
    results.append({
        'test_name': 'no double play with runner on 1B and 2 outs',
        'passed': game.double_play_count == 0 and game.inning == 2 and game.outs == 0,
        'expected': 'double_play_count=0, inning advanced to 2',
        'actual': f'double_play_count={game.double_play_count}, inning={game.inning}, outs={game.outs}'
    })

    game = FantasyGame(verbose=False)
    game.current_batter = {'name': 'Batter'}
    game.bases['3B'] = {'name': 'Runner 3B'}
    game.outs = 2
    game.process_event('out', 'Flyout', 'fly')
    results.append({
        'test_name': 'no sac fly with runner on 3B and 2 outs',
        'passed': game.sac_fly_count == 0 and game.score == 0 and game.inning == 2,
        'expected': 'sac_fly_count=0, score=0, inning advanced to 2',
        'actual': f'sac_fly_count={game.sac_fly_count}, score={game.score}, inning={game.inning}'
    })

    game = FantasyGame(verbose=False)
    game.current_batter = {'name': 'Walker'}
    game.bases['1B'] = {'name': 'Runner 1B'}
    game.bases['2B'] = {'name': 'Runner 2B'}
    game.bases['3B'] = {'name': 'Runner 3B'}
    game.process_event('walk', 'Walk', 'other')
    results.append({
        'test_name': 'bases loaded walk forces in one run',
        'passed': game.score == 1 and game.bases['1B']['name'] == 'Walker',
        'expected': 'score=1 and batter on 1B',
        'actual': f"score={game.score}, batter_on_1B={game.bases['1B']['name']}"
    })

    game = FantasyGame(verbose=False)
    game.current_batter = {'name': 'Hitter'}
    game.bases['2B'] = {'name': 'Runner 2B'}
    game.process_event('single', 'Single', 'other')
    results.append({
        'test_name': 'runner on 2B scores on single',
        'passed': game.score == 1 and game.bases['1B']['name'] == 'Hitter',
        'expected': 'score=1 and batter on 1B',
        'actual': f"score={game.score}, batter_on_1B={game.bases['1B']['name']}"
    })

    selected_players = build_fixture_selected_players(include_ph=True)
    fixture_rows = [
        {'game_id': 'g1', 'sequence_number': 1, 'player_id': 'ph1', 'result_raw': 'Single', 'description': 'single', 'event_type': 'single', 'out_kind': None, 'inning': 'Top 1'},
        {'game_id': 'g1', 'sequence_number': 2, 'player_id': 's2', 'result_raw': 'Strikeout', 'description': 'strikeout', 'event_type': 'out', 'out_kind': 'other', 'inning': 'Top 1'},
        {'game_id': 'g1', 'sequence_number': 3, 'player_id': 's3', 'result_raw': 'Strikeout', 'description': 'strikeout', 'event_type': 'out', 'out_kind': 'other', 'inning': 'Top 1'}
    ]
    fixture_df = pd.DataFrame(fixture_rows)
    result = run_single_simulation(selected_players, fixture_df, verbose=False)
    subbed = any('Subbing in Bench 1 for Starter 1 at C.' in line for line in result['game'].log)
    results.append({
        'test_name': 'substitution occurs when starter has no plays and PH is available',
        'passed': subbed,
        'expected': 'substitution log entry present',
        'actual': str(subbed)
    })

    selected_players = build_fixture_selected_players(include_ph=False)
    empty_fixture_df = pd.DataFrame([
        {'game_id': 'g2', 'sequence_number': 1, 'player_id': 'ghost', 'result_raw': 'Single', 'description': 'single', 'event_type': 'single', 'out_kind': None, 'inning': 'Top 1'}
    ])
    result = run_single_simulation(selected_players, empty_fixture_df, verbose=False)
    results.append({
        'test_name': 'dead lineup before inning 9 is flagged incomplete',
        'passed': result['completed_nine'] is False and result['termination_reason'] == 'no_usable_batters',
        'expected': 'completed_nine=False and termination_reason=no_usable_batters',
        'actual': f"completed_nine={result['completed_nine']}, termination_reason={result['termination_reason']}"
    })

    return pd.DataFrame(results)


targeted_checks_df = run_targeted_engine_checks()
display(targeted_checks_df)
print(f"Passed {int(targeted_checks_df['passed'].sum())} of {len(targeted_checks_df)} targeted checks.")


,test_name,passed,expected,actual
0,sac fly with runner on 3B and 0 outs,True,"sac_fly_count=1, score=1, outs=1","sac_fly_count=1, score=1, outs=1"
1,double play with runner on 1B and 0 outs,True,"double_play_count=1, outs=2","double_play_count=1, outs=2"
2,no double play with runner on 1B and 2 outs,True,"double_play_count=0, inning advanced to 2","double_play_count=0, inning=2, outs=0"
3,no sac fly with runner on 3B and 2 outs,True,"sac_fly_count=0, score=0, inning advanced to 2","sac_fly_count=0, score=0, inning=2"
4,bases loaded walk forces in one run,True,score=1 and batter on 1B,"score=1, batter_on_1B=Walker"
5,runner on 2B scores on single,True,score=1 and batter on 1B,"score=1, batter_on_1B=Hitter"
6,substitution occurs when starter has no plays ...,True,substitution log entry present,True
7,dead lineup before inning 9 is flagged incomplete,True,completed_nine=False and termination_reason=no...,"completed_nine=False, termination_reason=no_us..."


Passed 8 of 8 targeted checks.
